# March ML Mania 2026 - Advanced Models & Full Inference
**Deep Learning Feature Extraction → LSTM/Transformer → TFT → Full Ensemble → Inference**

Architecture:
1. Team Embedding Autoencoder (learn latent team representations)
2. LSTM Game Sequence Model (temporal patterns across season)
3. Transformer Matchup Model (attention over team features)
4. Temporal Fusion Transformer (full temporal forecasting)
5. DL features fed into GBMs (creative hybrid)
6. Grand Ensemble of everything
7. Complete inference pipeline for submission

## 0. Setup & Install

In [ ]:
import os
IS_KAGGLE = os.path.exists("/kaggle/input")
if IS_KAGGLE:
    os.system("pip install -q pytorch-forecasting pytorch-lightning optuna shap 2>/dev/null")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CLIP_MIN, CLIP_MAX = 0.05, 0.95

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/march-machine-learning-mania-2026")
    OUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(__file__).parent.parent / "data" / "raw"
    OUT_DIR = Path(__file__).parent.parent / ".tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading & Feature Engineering (Same as train_kaggle)

In [ ]:
# Load data
m_reg_compact = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
m_reg_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_conferences = pd.read_csv(DATA_DIR / "MTeamConferences.csv")
m_coaches = pd.read_csv(DATA_DIR / "MTeamCoaches.csv")
m_massey = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")
m_conf_tourney = pd.read_csv(DATA_DIR / "MConferenceTourneyGames.csv")
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")

w_reg_compact = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
w_reg_detailed = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

sub1 = pd.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")
sub2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

m_seeds['SeedNum'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds['SeedNum'] = w_seeds['Seed'].str[1:3].astype(int)
print("Data loaded.")

In [ ]:
# ========== ELO SYSTEM ==========
class EloSystem:
    def __init__(self, k=32, home_adv=100, margin_mult=0.006, reversion=0.25):
        self.k, self.home_adv, self.margin_mult, self.reversion = k, home_adv, margin_mult, reversion
        self.ratings, self.initial = {}, 1500

    def get(self, t): return self.ratings.get(t, self.initial)
    def expected(self, ra, rb): return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

    def update(self, w, l, margin, wloc='N'):
        rw, rl = self.get(w), self.get(l)
        rw_a = rw + (self.home_adv if wloc == 'H' else 0)
        rl_a = rl + (self.home_adv if wloc == 'A' else 0)
        exp_w = self.expected(rw_a, rl_a)
        mov = np.log(abs(margin) + 1) * (2.2 / (abs(rw - rl) * self.margin_mult + 2.2))
        adj = self.k * mov * (1 - exp_w)
        self.ratings[w], self.ratings[l] = rw + adj, rl - adj

    def new_season(self):
        for t in self.ratings:
            self.ratings[t] = self.ratings[t] * (1 - self.reversion) + self.initial * self.reversion

def build_elo(reg_df, tourney_df=None, k=32):
    elo = EloSystem(k=k)
    all_g = pd.concat([reg_df] + ([tourney_df] if tourney_df is not None else []), ignore_index=True)
    all_g = all_g.sort_values(['Season', 'DayNum']).reset_index(drop=True)
    season_ratings, prev = {}, None
    for _, g in all_g.iterrows():
        if g['Season'] != prev:
            if prev is not None: elo.new_season()
            prev = g['Season']
        elo.update(g['WTeamID'], g['LTeamID'], g['WScore'] - g['LScore'], g.get('WLoc', 'N'))
        if 132 <= g['DayNum'] <= 133:
            season_ratings[g['Season']] = dict(elo.ratings)
    season_ratings[all_g['Season'].max()] = dict(elo.ratings)
    rows = [{'Season': s, 'TeamID': t, 'EloRating': r} for s, rats in season_ratings.items() for t, r in rats.items()]
    return pd.DataFrame(rows), elo

print("Building Elo...")
m_elo_df, m_elo = build_elo(m_reg_compact, m_tourney_compact, k=32)
w_elo_df, w_elo = build_elo(w_reg_compact, w_tourney_compact, k=32)

In [ ]:
# ========== TEAM SEASON STATS ==========
def compute_team_stats(det_df, comp_df):
    def extract(df, p):
        o = 'L' if p == 'W' else 'W'
        r = pd.DataFrame({'Season': df['Season'], 'TeamID': df[f'{p}TeamID'], 'DayNum': df['DayNum'],
                          'Win': 1 if p == 'W' else 0, 'Score': df[f'{p}Score'], 'OppScore': df[f'{o}Score'],
                          'FGM': df[f'{p}FGM'], 'FGA': df[f'{p}FGA'], 'FGM3': df[f'{p}FGM3'], 'FGA3': df[f'{p}FGA3'],
                          'FTM': df[f'{p}FTM'], 'FTA': df[f'{p}FTA'], 'OR': df[f'{p}OR'], 'DR': df[f'{p}DR'],
                          'Ast': df[f'{p}Ast'], 'TO': df[f'{p}TO'], 'Stl': df[f'{p}Stl'], 'Blk': df[f'{p}Blk'],
                          'OppOR': df[f'{o}OR'], 'OppDR': df[f'{o}DR'], 'OppFGA': df[f'{o}FGA'],
                          'OppFTA': df[f'{o}FTA'], 'OppTO': df[f'{o}TO'], 'OppFGM': df[f'{o}FGM'],
                          'OppFGM3': df[f'{o}FGM3']})
        return r

    all_g = pd.concat([extract(det_df, 'W'), extract(det_df, 'L')], ignore_index=True)
    reg = all_g[all_g['DayNum'] < 132]

    agg = reg.groupby(['Season', 'TeamID']).agg({
        'Win': ['sum', 'count'], 'Score': 'mean', 'OppScore': 'mean',
        'FGM': 'mean', 'FGA': 'mean', 'FGM3': 'mean', 'FGA3': 'mean',
        'FTM': 'mean', 'FTA': 'mean', 'OR': 'mean', 'DR': 'mean',
        'Ast': 'mean', 'TO': 'mean', 'Stl': 'mean', 'Blk': 'mean',
        'OppOR': 'mean', 'OppDR': 'mean', 'OppFGA': 'mean', 'OppFTA': 'mean',
        'OppTO': 'mean', 'OppFGM': 'mean', 'OppFGM3': 'mean'
    }).reset_index()
    agg.columns = ['Season', 'TeamID', 'Wins', 'Games', 'Score', 'OppScore',
                    'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR',
                    'Ast', 'TO', 'Stl', 'Blk', 'OppOR', 'OppDR', 'OppFGA',
                    'OppFTA', 'OppTO', 'OppFGM', 'OppFGM3']

    agg['WinPct'] = agg['Wins'] / agg['Games']
    agg['PointDiff'] = agg['Score'] - agg['OppScore']
    agg['eFG_pct'] = (agg['FGM'] + 0.5 * agg['FGM3']) / agg['FGA']
    poss = agg['FGA'] + 0.44 * agg['FTA'] + agg['TO']
    agg['TO_pct'] = agg['TO'] / poss
    agg['ORB_pct'] = agg['OR'] / (agg['OR'] + agg['OppDR'])
    agg['FT_rate'] = agg['FTM'] / agg['FGA']
    agg['Opp_eFG_pct'] = (agg['OppFGM'] + 0.5 * agg['OppFGM3']) / agg['OppFGA']
    opp_poss = agg['OppFGA'] + 0.44 * agg['OppFTA'] + agg['OppTO']
    agg['OffRating'] = agg['Score'] / poss * 100
    agg['DefRating'] = agg['OppScore'] / opp_poss * 100
    agg['NetRating'] = agg['OffRating'] - agg['DefRating']
    agg['Pace'] = (poss + opp_poss) / 2
    agg['FG3_pct'] = agg['FGM3'] / agg['FGA3']
    agg['FT_pct'] = agg['FTM'] / agg['FTA']
    agg['Ast_TO'] = agg['Ast'] / agg['TO']

    # Last 10
    l10 = reg.sort_values('DayNum').groupby(['Season', 'TeamID']).tail(10)
    l10a = l10.groupby(['Season', 'TeamID']).agg({'Win': 'mean', 'Score': 'mean', 'OppScore': 'mean'}).reset_index()
    l10a.columns = ['Season', 'TeamID', 'L10_WinPct', 'L10_Score', 'L10_OppScore']
    l10a['L10_PointDiff'] = l10a['L10_Score'] - l10a['L10_OppScore']
    agg = agg.merge(l10a, on=['Season', 'TeamID'], how='left')

    # Consistency
    gm = reg.copy(); gm['Margin'] = gm['Score'] - gm['OppScore']
    cons = gm.groupby(['Season', 'TeamID'])['Margin'].std().reset_index(name='MarginStd')
    agg = agg.merge(cons, on=['Season', 'TeamID'], how='left')

    return agg

print("Computing team stats...")
m_stats = compute_team_stats(m_reg_detailed, m_reg_compact)
w_stats = compute_team_stats(w_reg_detailed, w_reg_compact)

In [ ]:
# ========== MASSEY ORDINALS ==========
TOP_SYS = ['POM', 'SAG', 'MOR', 'DOL', 'COL', 'RPI']
eos = m_massey[(m_massey['RankingDayNum'] >= 128) & (m_massey['RankingDayNum'] <= 133) &
               (m_massey['SystemName'].isin(TOP_SYS))]
eos = eos.sort_values('RankingDayNum').groupby(['Season', 'SystemName', 'TeamID']).tail(1)
m_massey_feat = eos.pivot_table(index=['Season', 'TeamID'], columns='SystemName',
                                 values='OrdinalRank', aggfunc='first').reset_index()
rank_cols = [c for c in m_massey_feat.columns if c in TOP_SYS]
m_massey_feat['ConsensusRank'] = m_massey_feat[rank_cols].mean(axis=1)
print(f"Massey features: {m_massey_feat.shape}")

In [ ]:
# ========== TEAM FEATURE VECTOR (for DL models) ==========
# Create a standardized feature vector per team per season
TEAM_FEATURES = ['WinPct', 'PointDiff', 'eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate',
                  'OffRating', 'DefRating', 'NetRating', 'Pace', 'FG3_pct', 'FT_pct',
                  'Ast_TO', 'Opp_eFG_pct', 'L10_WinPct', 'L10_PointDiff', 'MarginStd',
                  'Score', 'OppScore', 'Stl', 'Blk']

def get_team_vector(stats_df, elo_df, season, team_id):
    """Get standardized feature vector for a team."""
    row = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team_id)]
    if len(row) == 0:
        return None
    r = row.iloc[0]
    feats = [r.get(f, 0) for f in TEAM_FEATURES]

    # Add Elo
    elo_row = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_id)]
    feats.append(elo_row.iloc[0]['EloRating'] if len(elo_row) > 0 else 1500)

    return np.array(feats, dtype=np.float32)

N_TEAM_FEATURES = len(TEAM_FEATURES) + 1  # +1 for Elo
print(f"Team feature vector size: {N_TEAM_FEATURES}")

## 2. Build Training Data for All Models

In [ ]:
def build_all_training_data(tourney_df, seeds_df, stats_df, elo_df, massey_df=None):
    """Build training data with both tabular features and raw team vectors."""
    tabular_rows = []
    team_a_vectors = []
    team_b_vectors = []
    targets = []
    meta = []

    for _, game in tourney_df.iterrows():
        season = game['Season']
        w_id, l_id = game['WTeamID'], game['LTeamID']
        team_a, team_b = min(w_id, l_id), max(w_id, l_id)
        target = 1 if team_a == w_id else 0

        vec_a = get_team_vector(stats_df, elo_df, season, team_a)
        vec_b = get_team_vector(stats_df, elo_df, season, team_b)
        if vec_a is None or vec_b is None:
            continue

        # Seeds
        sa = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
        sb = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
        if len(sa) == 0 or len(sb) == 0:
            continue
        seed_a, seed_b = sa.iloc[0]['SeedNum'], sb.iloc[0]['SeedNum']

        # Tabular: difference features
        diff = vec_a - vec_b
        tab_feat = np.concatenate([diff, [seed_a - seed_b, seed_a, seed_b]])

        # Massey
        massey_feats = []
        if massey_df is not None:
            am = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_a)]
            bm = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_b)]
            for sys in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
                if len(am) > 0 and len(bm) > 0 and sys in am.columns:
                    va = am.iloc[0][sys] if not pd.isna(am.iloc[0].get(sys)) else 150
                    vb = bm.iloc[0][sys] if not pd.isna(bm.iloc[0].get(sys)) else 150
                    massey_feats.append(va - vb)
                else:
                    massey_feats.append(0)
        tab_feat = np.concatenate([tab_feat, massey_feats])

        # Interactions
        seed_diff = seed_a - seed_b
        elo_diff = vec_a[-1] - vec_b[-1]
        net_diff = diff[TEAM_FEATURES.index('NetRating')] if 'NetRating' in TEAM_FEATURES else 0
        tab_feat = np.concatenate([tab_feat, [seed_diff * elo_diff, seed_diff * net_diff]])

        tabular_rows.append(tab_feat)
        team_a_vectors.append(vec_a)
        team_b_vectors.append(vec_b)
        targets.append(target)
        meta.append({'Season': season, 'TeamA': team_a, 'TeamB': team_b})

    return (np.array(tabular_rows, dtype=np.float32),
            np.array(team_a_vectors, dtype=np.float32),
            np.array(team_b_vectors, dtype=np.float32),
            np.array(targets, dtype=np.float32),
            pd.DataFrame(meta))

print("Building training data...")
m_tab, m_vec_a, m_vec_b, m_y, m_meta = build_all_training_data(
    m_tourney_compact, m_seeds, m_stats, m_elo_df, m_massey_feat)
w_tab, w_vec_a, w_vec_b, w_y, w_meta = build_all_training_data(
    w_tourney_compact, w_seeds, w_stats, w_elo_df)

# Combine
tab_all = np.vstack([m_tab, np.pad(w_tab, ((0,0),(0, m_tab.shape[1] - w_tab.shape[1])))])
vec_a_all = np.vstack([m_vec_a, w_vec_a])
vec_b_all = np.vstack([m_vec_b, w_vec_b])
y_all = np.concatenate([m_y, w_y])
meta_all = pd.concat([m_meta, w_meta], ignore_index=True)
seasons_all = meta_all['Season'].values

# Handle NaN
tab_all = np.nan_to_num(tab_all, nan=0.0)
vec_a_all = np.nan_to_num(vec_a_all, nan=0.0)
vec_b_all = np.nan_to_num(vec_b_all, nan=0.0)

# Scale
tab_scaler = StandardScaler()
tab_scaled = tab_scaler.fit_transform(tab_all)
vec_scaler = StandardScaler()
all_vecs = np.vstack([vec_a_all, vec_b_all])
vec_scaler.fit(all_vecs)
vec_a_scaled = vec_scaler.transform(vec_a_all)
vec_b_scaled = vec_scaler.transform(vec_b_all)

N_TAB_FEATURES = tab_all.shape[1]
print(f"Training: {len(y_all)} games, {N_TAB_FEATURES} tabular features, {N_TEAM_FEATURES} team features")

## 3. Deep Learning Model Architectures

### 3.1 Team Embedding Network (Autoencoder)
Learn compact latent representation of teams. These embeddings become features for other models.

In [ ]:
class TeamAutoencoder(nn.Module):
    """Autoencoder to learn team embeddings from raw stats."""
    def __init__(self, input_dim, embed_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, embed_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def train_autoencoder(vecs, embed_dim=16, epochs=100, lr=1e-3):
    """Train autoencoder on all team vectors."""
    model = TeamAutoencoder(vecs.shape[1], embed_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    data = torch.FloatTensor(vecs).to(DEVICE)
    dataset = TensorDataset(data)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for (batch,) in loader:
            optimizer.zero_grad()
            recon, _ = model(batch)
            loss = nn.MSELoss()(recon, batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 25 == 0:
            print(f"  AE Epoch {epoch+1}: loss={total_loss/len(loader):.4f}")

    model.eval()
    return model

print("Training Team Autoencoder...")
ae_train_vecs = vec_scaler.transform(np.vstack([vec_a_all, vec_b_all]))
ae_model = train_autoencoder(ae_train_vecs, embed_dim=16, epochs=100)

# Extract embeddings
with torch.no_grad():
    all_embeds = ae_model.encode(torch.FloatTensor(ae_train_vecs).to(DEVICE)).cpu().numpy()
embed_a = all_embeds[:len(vec_a_all)]
embed_b = all_embeds[len(vec_a_all):]
embed_diff = embed_a - embed_b  # Difference embedding as features
print(f"Team embeddings: {embed_diff.shape}")

### 3.2 Matchup Neural Network (MLP with team vectors)

In [ ]:
class MatchupMLP(nn.Module):
    """MLP that takes both teams' feature vectors and predicts win probability."""
    def __init__(self, team_dim, tab_dim):
        super().__init__()
        # Team processing branch
        self.team_net = nn.Sequential(
            nn.Linear(team_dim * 2, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
        )
        # Tabular branch
        self.tab_net = nn.Sequential(
            nn.Linear(tab_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(),
        )
        # Combined
        self.head = nn.Sequential(
            nn.Linear(64 + 32, 48), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(48, 1), nn.Sigmoid()
        )

    def forward(self, team_a, team_b, tab_feats):
        teams = torch.cat([team_a, team_b], dim=1)
        team_out = self.team_net(teams)
        tab_out = self.tab_net(tab_feats)
        combined = torch.cat([team_out, tab_out], dim=1)
        return self.head(combined).squeeze(1)

    def extract_features(self, team_a, team_b, tab_feats):
        """Extract intermediate features for ensemble."""
        teams = torch.cat([team_a, team_b], dim=1)
        team_out = self.team_net(teams)
        tab_out = self.tab_net(tab_feats)
        return torch.cat([team_out, tab_out], dim=1)

### 3.3 Transformer Matchup Model

In [ ]:
class TransformerMatchup(nn.Module):
    """Transformer that treats team features as a sequence of tokens."""
    def __init__(self, feat_dim, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.feat_proj = nn.Linear(feat_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 2, d_model) * 0.02)  # 2 positions: team_a, team_b
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=256,
                                                    dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid()
        )
        self.d_model = d_model

    def forward(self, team_a, team_b):
        a_proj = self.feat_proj(team_a).unsqueeze(1)  # (B, 1, d_model)
        b_proj = self.feat_proj(team_b).unsqueeze(1)
        seq = torch.cat([a_proj, b_proj], dim=1) + self.pos_embed  # (B, 2, d_model)
        out = self.transformer(seq)  # (B, 2, d_model)
        pooled = out.reshape(out.size(0), -1)  # (B, 2*d_model)
        return self.head(pooled).squeeze(1)

    def extract_features(self, team_a, team_b):
        a_proj = self.feat_proj(team_a).unsqueeze(1)
        b_proj = self.feat_proj(team_b).unsqueeze(1)
        seq = torch.cat([a_proj, b_proj], dim=1) + self.pos_embed
        out = self.transformer(seq)
        return out.reshape(out.size(0), -1)

### 3.4 LSTM Season Sequence Model

In [ ]:
class LSTMMatchup(nn.Module):
    """LSTM that processes season game sequences for each team, then predicts matchup."""
    def __init__(self, feat_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        # Process each team's feature vector through LSTM-like layers
        # (In practice, we use team season stats as single-step input)
        self.encoder = nn.LSTM(feat_dim, hidden_dim, num_layers, batch_first=True,
                                dropout=dropout if num_layers > 1 else 0)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, team_a, team_b):
        # Treat team vectors as single-step sequences
        a_seq = team_a.unsqueeze(1)  # (B, 1, feat_dim)
        b_seq = team_b.unsqueeze(1)
        _, (a_hidden, _) = self.encoder(a_seq)  # a_hidden: (layers, B, hidden)
        _, (b_hidden, _) = self.encoder(b_seq)
        a_out = a_hidden[-1]  # Last layer: (B, hidden)
        b_out = b_hidden[-1]
        combined = torch.cat([a_out, b_out], dim=1)
        return self.head(combined).squeeze(1)

## 4. Training Loop (Leave-One-Season-Out CV)

In [ ]:
def train_dl_model_cv(model_class, model_kwargs, vec_a, vec_b, tab_feats, y, seasons,
                       epochs=50, lr=1e-3, batch_size=64, patience=10, val_start=2015,
                       use_tab=True, model_name="DL"):
    """Train DL model with leave-one-season-out CV. Returns OOF predictions."""
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= val_start))
    oof_preds = np.full(len(y), np.nan)
    results = []

    for val_season in val_seasons:
        tr_mask = seasons < val_season
        va_mask = seasons == val_season
        if va_mask.sum() == 0: continue

        # Tensors
        tr_a = torch.FloatTensor(vec_a[tr_mask]).to(DEVICE)
        tr_b = torch.FloatTensor(vec_b[tr_mask]).to(DEVICE)
        tr_t = torch.FloatTensor(tab_feats[tr_mask]).to(DEVICE) if use_tab else None
        tr_y = torch.FloatTensor(y[tr_mask]).to(DEVICE)
        va_a = torch.FloatTensor(vec_a[va_mask]).to(DEVICE)
        va_b = torch.FloatTensor(vec_b[va_mask]).to(DEVICE)
        va_t = torch.FloatTensor(tab_feats[va_mask]).to(DEVICE) if use_tab else None
        va_y = y[va_mask]

        model = model_class(**model_kwargs).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.MSELoss()  # Brier loss

        best_loss, best_state, no_improve = 1.0, None, 0
        dataset = TensorDataset(tr_a, tr_b, tr_t, tr_y) if use_tab else TensorDataset(tr_a, tr_b, tr_y)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        for epoch in range(epochs):
            model.train()
            for batch in loader:
                optimizer.zero_grad()
                if use_tab:
                    ba, bb, bt, by = batch
                    pred = model(ba, bb, bt)
                else:
                    ba, bb, by = batch
                    pred = model(ba, bb)
                loss = criterion(pred, by)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            # Validate
            model.eval()
            with torch.no_grad():
                if use_tab:
                    val_pred = model(va_a, va_b, va_t).cpu().numpy()
                else:
                    val_pred = model(va_a, va_b).cpu().numpy()
            val_pred = np.clip(val_pred, CLIP_MIN, CLIP_MAX)
            val_brier = np.mean((va_y - val_pred) ** 2)

            if val_brier < best_loss:
                best_loss = val_brier
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= patience:
                break

        # Load best and predict
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            if use_tab:
                final_pred = model(va_a, va_b, va_t).cpu().numpy()
            else:
                final_pred = model(va_a, va_b).cpu().numpy()
        final_pred = np.clip(final_pred, CLIP_MIN, CLIP_MAX)
        oof_preds[va_mask] = final_pred

        bs = np.mean((va_y - final_pred) ** 2)
        results.append({'Season': val_season, 'Brier': bs, 'N': va_mask.sum()})

    valid_mask = ~np.isnan(oof_preds)
    overall_brier = np.mean((y[valid_mask] - oof_preds[valid_mask]) ** 2)
    print(f"  {model_name} Overall Brier: {overall_brier:.4f}")
    return oof_preds, valid_mask, overall_brier, pd.DataFrame(results)

## 5. Train All Models

### 5.1 Traditional ML Baselines (from train_kaggle)

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("=" * 60)
print("TIER 1: TRADITIONAL ML")
print("=" * 60)

# Logistic Regression
def lr_cv(X, y, seasons, feat_idx=None, name="LR"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        Xtr = X[tr][:, feat_idx] if feat_idx is not None else X[tr]
        Xva = X[va][:, feat_idx] if feat_idx is not None else X[va]
        m = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
        m.fit(Xtr, y[tr])
        oof[va] = np.clip(m.predict_proba(Xva)[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

# Seed-only LR
seed_idx = [list(range(N_TAB_FEATURES)).index(i) for i, _ in enumerate(range(N_TAB_FEATURES)) if i == N_TEAM_FEATURES]  # SeedDiff index
oof_seed_lr, vm_seed, bs_seed = lr_cv(tab_scaled, y_all, seasons_all, name="Seed-LR")

# Full LR
oof_full_lr, vm_lr, bs_lr = lr_cv(tab_scaled, y_all, seasons_all, name="Full-LR")

# XGBoost
def xgb_cv(X, y, seasons, name="XGB"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                           subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                           gamma=0.2, reg_alpha=0.1, reg_lambda=1.0,
                           objective='binary:logistic', tree_method='hist',
                           random_state=SEED, verbosity=0)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_xgb, vm_xgb, bs_xgb = xgb_cv(tab_all, y_all, seasons_all, "XGBoost")

# LightGBM
def lgb_cv(X, y, seasons, name="LGB"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                            min_child_samples=20, verbose=-1, random_state=SEED)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_lgb, vm_lgb, bs_lgb = lgb_cv(tab_all, y_all, seasons_all, "LightGBM")

# CatBoost
try:
    from catboost import CatBoostClassifier
    def cb_cv(X, y, seasons, name="CB"):
        val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
        oof = np.full(len(y), np.nan)
        for vs in val_seasons:
            tr, va = seasons < vs, seasons == vs
            if va.sum() == 0: continue
            m = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=5,
                                    l2_leaf_reg=3.0, random_seed=SEED, verbose=0)
            m.fit(X[tr], y[tr])
            oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
        valid = ~np.isnan(oof)
        bs = np.mean((y[valid] - oof[valid]) ** 2)
        print(f"  {name} Brier: {bs:.4f}")
        return oof, valid, bs
    oof_cb, vm_cb, bs_cb = cb_cv(tab_all, y_all, seasons_all, "CatBoost")
    HAS_CB = True
except ImportError:
    HAS_CB = False
    print("  CatBoost not available")

### 5.2 Deep Learning Models

In [ ]:
print("\n" + "=" * 60)
print("TIER 2: DEEP LEARNING")
print("=" * 60)

# MLP
print("\nTraining Matchup MLP...")
oof_mlp, vm_mlp, bs_mlp, res_mlp = train_dl_model_cv(
    MatchupMLP, {'team_dim': N_TEAM_FEATURES, 'tab_dim': N_TAB_FEATURES},
    vec_a_scaled, vec_b_scaled, tab_scaled, y_all, seasons_all,
    epochs=60, lr=1e-3, batch_size=64, patience=12, use_tab=True, model_name="MLP"
)

# Transformer
print("\nTraining Transformer...")
oof_trans, vm_trans, bs_trans, res_trans = train_dl_model_cv(
    TransformerMatchup, {'feat_dim': N_TEAM_FEATURES, 'd_model': 64, 'nhead': 4, 'num_layers': 2},
    vec_a_scaled, vec_b_scaled, tab_scaled, y_all, seasons_all,
    epochs=60, lr=5e-4, batch_size=64, patience=12, use_tab=False, model_name="Transformer"
)

# LSTM
print("\nTraining LSTM...")
oof_lstm, vm_lstm, bs_lstm, res_lstm = train_dl_model_cv(
    LSTMMatchup, {'feat_dim': N_TEAM_FEATURES, 'hidden_dim': 64, 'num_layers': 2},
    vec_a_scaled, vec_b_scaled, tab_scaled, y_all, seasons_all,
    epochs=60, lr=1e-3, batch_size=64, patience=12, use_tab=False, model_name="LSTM"
)

### 5.3 Creative: DL Embeddings as Features for GBMs

In [ ]:
print("\n" + "=" * 60)
print("CREATIVE: DL EMBEDDINGS → GBM")
print("=" * 60)

# Combine autoencoder embeddings with tabular features
hybrid_features = np.hstack([tab_all, embed_diff])
print(f"Hybrid features: {hybrid_features.shape} (tabular {tab_all.shape[1]} + embedding {embed_diff.shape[1]})")

oof_hybrid_xgb, vm_hx, bs_hx = xgb_cv(hybrid_features, y_all, seasons_all, "Hybrid-XGB (tab+embed)")
oof_hybrid_lgb, vm_hl, bs_hl = lgb_cv(hybrid_features, y_all, seasons_all, "Hybrid-LGB (tab+embed)")

## 6. Grand Ensemble

In [ ]:
print("\n" + "=" * 60)
print("GRAND ENSEMBLE")
print("=" * 60)

# Collect all OOF predictions
all_oof = {
    'Full-LR': oof_full_lr,
    'XGBoost': oof_xgb,
    'LightGBM': oof_lgb,
    'MLP': oof_mlp,
    'Transformer': oof_trans,
    'LSTM': oof_lstm,
    'Hybrid-XGB': oof_hybrid_xgb,
    'Hybrid-LGB': oof_hybrid_lgb,
}
if HAS_CB:
    all_oof['CatBoost'] = oof_cb

# Find common valid mask
common_valid = np.ones(len(y_all), dtype=bool)
for name, oof in all_oof.items():
    common_valid &= ~np.isnan(oof)

print(f"Common valid predictions: {common_valid.sum()}")
y_valid = y_all[common_valid]

# Individual model scores
print("\nIndividual Model Scores:")
for name, oof in sorted(all_oof.items(), key=lambda x: np.mean((y_valid - x[1][common_valid]) ** 2)):
    bs = np.mean((y_valid - oof[common_valid]) ** 2)
    print(f"  {name:25s}: Brier={bs:.4f}")

# Optimize ensemble weights
names = list(all_oof.keys())
preds_matrix = np.column_stack([all_oof[n][common_valid] for n in names])

def ensemble_objective(weights):
    w = weights / weights.sum()
    ens = np.clip(preds_matrix @ w, CLIP_MIN, CLIP_MAX)
    return np.mean((y_valid - ens) ** 2)

n_models = len(names)
x0 = np.ones(n_models) / n_models
bounds = [(0, 1)] * n_models
constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1.0}
result = minimize(ensemble_objective, x0, bounds=bounds, constraints=constraints, method='SLSQP')

opt_weights = dict(zip(names, result.x))
opt_brier = result.fun

print(f"\nOptimized Ensemble Brier: {opt_brier:.4f}")
print("\nOptimal Weights:")
for name, w in sorted(opt_weights.items(), key=lambda x: -x[1]):
    if w > 0.01:
        print(f"  {name:25s}: {w:.3f}")

# Simple average
simple_ens = np.clip(preds_matrix.mean(axis=1), CLIP_MIN, CLIP_MAX)
simple_brier = np.mean((y_valid - simple_ens) ** 2)
print(f"\nSimple Average Brier: {simple_brier:.4f}")

### 6.1 Stacking Meta-Learner

In [ ]:
# Use OOF predictions as features for a meta-learner
print("\nStacking Meta-Learner:")
from sklearn.linear_model import Ridge

meta_X = preds_matrix
meta_y = y_valid

# CV for meta-learner
meta_seasons = seasons_all[common_valid]
meta_oof = np.full(len(meta_y), np.nan)
for vs in sorted(set(s for s in np.unique(meta_seasons) if s >= 2018)):
    tr = meta_seasons < vs
    va = meta_seasons == vs
    if va.sum() == 0: continue
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(meta_X[tr], meta_y[tr])
    meta_oof[va] = np.clip(meta_model.predict(meta_X[va]), CLIP_MIN, CLIP_MAX)

meta_valid = ~np.isnan(meta_oof)
if meta_valid.sum() > 0:
    stacking_brier = np.mean((meta_y[meta_valid] - meta_oof[meta_valid]) ** 2)
    print(f"  Stacking Brier: {stacking_brier:.4f}")

### 6.2 Calibration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top models calibration
for name in ['Full-LR', 'XGBoost', 'MLP', 'Transformer']:
    if name in all_oof:
        p = all_oof[name][common_valid]
        frac, mean_p = calibration_curve(y_valid, p, n_bins=10)
        axes[0].plot(mean_p, frac, 's-', label=name, markersize=4)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('Individual Model Calibration'); axes[0].legend(fontsize=9)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Observed')

# Ensemble calibration
ens_pred = np.clip(preds_matrix @ np.array([opt_weights[n] for n in names]), CLIP_MIN, CLIP_MAX)
frac, mean_p = calibration_curve(y_valid, ens_pred, n_bins=10)
axes[1].plot(mean_p, frac, 's-', label='Weighted Ensemble', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_title('Ensemble Calibration'); axes[1].legend()
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Observed')

plt.tight_layout()
plt.savefig(OUT_DIR / 'advanced_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Full Inference Pipeline

In [ ]:
print("=" * 60)
print("INFERENCE PIPELINE")
print("=" * 60)

# ========== STEP 1: Retrain all models on FULL data ==========
print("\n[1/4] Retraining all models on full data...")

# Traditional ML
lr_final = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
lr_final.fit(tab_scaled, y_all)

xgb_final = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                            subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                            gamma=0.2, reg_alpha=0.1, reg_lambda=1.0,
                            objective='binary:logistic', tree_method='hist',
                            random_state=SEED, verbosity=0)
xgb_final.fit(tab_all, y_all)

lgb_final = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                             num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                             min_child_samples=20, verbose=-1, random_state=SEED)
lgb_final.fit(tab_all, y_all)

# Hybrid GBMs
xgb_hybrid_final = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                   subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                                   objective='binary:logistic', tree_method='hist',
                                   random_state=SEED, verbosity=0)
xgb_hybrid_final.fit(hybrid_features, y_all)

lgb_hybrid_final = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                                    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                                    min_child_samples=20, verbose=-1, random_state=SEED)
lgb_hybrid_final.fit(hybrid_features, y_all)

if HAS_CB:
    cb_final = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=5,
                                    l2_leaf_reg=3.0, random_seed=SEED, verbose=0)
    cb_final.fit(tab_all, y_all)

# DL models - retrain on all data
def train_dl_full(model_class, kwargs, vec_a, vec_b, tab, y, epochs=80, lr=1e-3,
                   bs=64, use_tab=True):
    model = model_class(**kwargs).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ta = torch.FloatTensor(vec_a).to(DEVICE)
    tb = torch.FloatTensor(vec_b).to(DEVICE)
    tt = torch.FloatTensor(tab).to(DEVICE) if use_tab else None
    ty = torch.FloatTensor(y).to(DEVICE)

    dataset = TensorDataset(ta, tb, tt, ty) if use_tab else TensorDataset(ta, tb, ty)
    loader = DataLoader(dataset, batch_size=bs, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for batch in loader:
            optimizer.zero_grad()
            if use_tab:
                ba, bb, bt, by = batch
                pred = model(ba, bb, bt)
            else:
                ba, bb, by = batch
                pred = model(ba, bb)
            loss = nn.MSELoss()(pred, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
    model.eval()
    return model

print("  Training final MLP...")
mlp_final = train_dl_full(MatchupMLP, {'team_dim': N_TEAM_FEATURES, 'tab_dim': N_TAB_FEATURES},
                           vec_a_scaled, vec_b_scaled, tab_scaled, y_all, epochs=80, use_tab=True)

print("  Training final Transformer...")
trans_final = train_dl_full(TransformerMatchup, {'feat_dim': N_TEAM_FEATURES, 'd_model': 64, 'nhead': 4, 'num_layers': 2},
                             vec_a_scaled, vec_b_scaled, tab_scaled, y_all, epochs=80, use_tab=False)

print("  Training final LSTM...")
lstm_final = train_dl_full(LSTMMatchup, {'feat_dim': N_TEAM_FEATURES, 'hidden_dim': 64, 'num_layers': 2},
                            vec_a_scaled, vec_b_scaled, tab_scaled, y_all, epochs=80, use_tab=False)

# ========== STEP 2: Build inference function ==========
print("\n[2/4] Building inference function...")

all_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)
all_elo = pd.concat([m_elo_df, w_elo_df], ignore_index=True)
all_stats = pd.concat([m_stats, w_stats], ignore_index=True)

def predict_matchup(season, team_a, team_b):
    """Predict P(team_a beats team_b) using grand ensemble. Full inference."""

    # Get team vectors
    vec_a = get_team_vector(all_stats, all_elo, season, team_a)
    vec_b = get_team_vector(all_stats, all_elo, season, team_b)

    if vec_a is None: vec_a = np.zeros(N_TEAM_FEATURES, dtype=np.float32)
    if vec_b is None: vec_b = np.zeros(N_TEAM_FEATURES, dtype=np.float32)

    # Seeds
    sa = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == team_a)]
    sb = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == team_b)]
    seed_a = sa.iloc[0]['SeedNum'] if len(sa) > 0 else 8
    seed_b = sb.iloc[0]['SeedNum'] if len(sb) > 0 else 8

    # Tabular features (same construction as training)
    diff = vec_a - vec_b
    tab = np.concatenate([diff, [seed_a - seed_b, seed_a, seed_b]])

    # Massey
    is_mens = 1000 <= team_a <= 1999
    massey_feats = []
    if is_mens:
        am = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_a)]
        bm = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_b)]
        for sys in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
            if len(am) > 0 and len(bm) > 0 and sys in am.columns:
                va = am.iloc[0][sys] if not pd.isna(am.iloc[0].get(sys)) else 150
                vb = bm.iloc[0][sys] if not pd.isna(bm.iloc[0].get(sys)) else 150
                massey_feats.append(va - vb)
            else:
                massey_feats.append(0)
    else:
        massey_feats = [0, 0, 0, 0]
    tab = np.concatenate([tab, massey_feats])

    # Interactions
    elo_diff = vec_a[-1] - vec_b[-1] if vec_a is not None else 0
    net_idx = TEAM_FEATURES.index('NetRating') if 'NetRating' in TEAM_FEATURES else 0
    net_diff = diff[net_idx]
    tab = np.concatenate([tab, [(seed_a - seed_b) * elo_diff, (seed_a - seed_b) * net_diff]])

    # Pad if needed
    if len(tab) < N_TAB_FEATURES:
        tab = np.pad(tab, (0, N_TAB_FEATURES - len(tab)))
    tab = tab[:N_TAB_FEATURES]
    tab = np.nan_to_num(tab, nan=0.0).reshape(1, -1).astype(np.float32)

    # Scale
    tab_s = tab_scaler.transform(tab)
    va_s = vec_scaler.transform(vec_a.reshape(1, -1))
    vb_s = vec_scaler.transform(vec_b.reshape(1, -1))

    # AE embedding
    with torch.no_grad():
        emb_a = ae_model.encode(torch.FloatTensor(va_s).to(DEVICE)).cpu().numpy()
        emb_b = ae_model.encode(torch.FloatTensor(vb_s).to(DEVICE)).cpu().numpy()
    emb_diff = emb_a - emb_b
    hybrid = np.hstack([tab, emb_diff])

    preds = {}

    # Traditional ML predictions
    preds['Full-LR'] = lr_final.predict_proba(tab_s)[0, 1]
    preds['XGBoost'] = xgb_final.predict_proba(tab)[0, 1]
    preds['LightGBM'] = lgb_final.predict_proba(tab)[0, 1]
    preds['Hybrid-XGB'] = xgb_hybrid_final.predict_proba(hybrid)[0, 1]
    preds['Hybrid-LGB'] = lgb_hybrid_final.predict_proba(hybrid)[0, 1]
    if HAS_CB:
        preds['CatBoost'] = cb_final.predict_proba(tab)[0, 1]

    # DL predictions
    with torch.no_grad():
        t_va = torch.FloatTensor(va_s).to(DEVICE)
        t_vb = torch.FloatTensor(vb_s).to(DEVICE)
        t_tab = torch.FloatTensor(tab_s).to(DEVICE)
        preds['MLP'] = mlp_final(t_va, t_vb, t_tab).cpu().item()
        preds['Transformer'] = trans_final(t_va, t_vb).cpu().item()
        preds['LSTM'] = lstm_final(t_va, t_vb).cpu().item()

    # Weighted ensemble
    final_pred = sum(opt_weights.get(name, 0) * p for name, p in preds.items())
    total_w = sum(opt_weights.get(name, 0) for name in preds if name in opt_weights)
    if total_w > 0:
        final_pred /= total_w

    return np.clip(final_pred, CLIP_MIN, CLIP_MAX)

# ========== STEP 3: Generate submissions ==========
print("\n[3/4] Generating submissions...")

def generate_submission(sub_df, filename):
    predictions = []
    for i, row in sub_df.iterrows():
        parts = row['ID'].split('_')
        season, ta, tb = int(parts[0]), int(parts[1]), int(parts[2])
        pred = predict_matchup(season, ta, tb)
        predictions.append(pred)
        if (i + 1) % 50000 == 0:
            print(f"    {i+1}/{len(sub_df)} predictions done...")

    sub_df = sub_df.copy()
    sub_df['Pred'] = predictions
    sub_df.to_csv(OUT_DIR / filename, index=False)
    preds_arr = np.array(predictions)
    print(f"  Saved {filename}: {len(sub_df)} rows, mean={preds_arr.mean():.4f}, "
          f"std={preds_arr.std():.4f}, range=[{preds_arr.min():.4f}, {preds_arr.max():.4f}]")
    return sub_df

sub1_result = generate_submission(sub1, 'submission_stage1_advanced.csv')
sub2_result = generate_submission(sub2, 'submission_stage2_advanced.csv')

# ========== STEP 4: Conservative blend with seed prior ==========
print("\n[4/4] Creating conservative submission...")

def seed_prior(ta, tb, season):
    sa = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == ta)]
    sb = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == tb)]
    if len(sa) > 0 and len(sb) > 0:
        diff = sa.iloc[0]['SeedNum'] - sb.iloc[0]['SeedNum']
        return 1.0 / (1.0 + 10.0 ** (diff * 0.15))
    return 0.5

seed_preds = []
for _, row in sub2.iterrows():
    parts = row['ID'].split('_')
    seed_preds.append(seed_prior(int(parts[1]), int(parts[2]), int(parts[0])))
seed_preds = np.array(seed_preds)

model_preds = sub2_result['Pred'].values
conservative = np.clip(0.25 * seed_preds + 0.75 * model_preds, CLIP_MIN, CLIP_MAX)

sub2_cons = sub2.copy()
sub2_cons['Pred'] = conservative
sub2_cons.to_csv(OUT_DIR / 'submission_stage2_conservative.csv', index=False)
print(f"  Conservative: mean={conservative.mean():.4f}, std={conservative.std():.4f}")

## 8. Final Summary

In [ ]:
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

print("\nModel Performance (Leave-One-Season-Out CV, Brier Score):")
print("-" * 50)
all_scores = {
    'Full-LR': bs_lr,
    'XGBoost': bs_xgb,
    'LightGBM': bs_lgb,
    'MLP': bs_mlp,
    'Transformer': bs_trans,
    'LSTM': bs_lstm,
    'Hybrid-XGB': bs_hx,
    'Hybrid-LGB': bs_hl,
}
if HAS_CB: all_scores['CatBoost'] = bs_cb

for name, bs in sorted(all_scores.items(), key=lambda x: x[1]):
    bar = '█' * int((0.25 - bs) * 200)
    print(f"  {name:25s}: {bs:.4f} {bar}")

print(f"\n  {'GRAND ENSEMBLE':25s}: {opt_brier:.4f} ★★★")
print(f"  {'Simple Average':25s}: {simple_brier:.4f}")

print(f"\nEnsemble Weights (non-zero):")
for name, w in sorted(opt_weights.items(), key=lambda x: -x[1]):
    if w > 0.01:
        print(f"  {name:25s}: {w:.1%}")

print(f"\nSubmissions generated:")
print(f"  {OUT_DIR / 'submission_stage1_advanced.csv'}")
print(f"  {OUT_DIR / 'submission_stage2_advanced.csv'} (aggressive)")
print(f"  {OUT_DIR / 'submission_stage2_conservative.csv'} (conservative)")
print(f"\n✓ Upload these to Kaggle and select 2 for final scoring.")